# Parrotlet-A 2.5 Pro — English Audio Transcription (Colab)

This notebook loads Eka Care's `parrotlet-a-2.5-pro` speech-LLM and transcribes an uploaded English audio recording.

**Fixes baked into this notebook:**
1. `transformers` pinned to a version compatible with the model's custom `Gemma3ForConditionalGeneration` decoder (avoids the `inputs_embeds` forwarding error).
2. A patched `transcribe_fixed()` that casts audio features to the encoder's dtype (bf16) instead of upcasting the whole model to fp32 — avoids the CUDA OOM you hit on a T4 (~14.5GB) when calling `model.float()`.
3. Skips the library's internal resampling bug by pre-resampling audio to 16kHz with `librosa` before calling transcribe.

**If your GPU disconnects mid-session:** re-run cells **1 → 4** in order (install → restart runtime if prompted → load model → upload+transcribe). You do NOT need to re-run cell 1's pip install every time unless you get a fresh Colab VM (not just a session/runtime reset) — but it's safe to re-run regardless.

## 1. Install / pin dependencies
Run once per fresh Colab VM. If prompted to restart the runtime after this, do so (Runtime → Restart runtime), then continue from cell 2.

In [ ]:
!pip install "transformers==4.52.0" --quiet
!pip install librosa soxr --quiet

import transformers
print("transformers version:", transformers.__version__)

## 2. Load the model
Run this after any runtime restart / GPU reconnect. Loads in bf16 automatically — do **not** call `.float()` on this model, it will OOM on a T4.

In [ ]:
from transformers import AutoModel

MODEL_ID = "ekacare/parrotlet-a-2.5-pro"

print("Loading Parrotlet-A...")

model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device_map="auto"
)

print("=" * 70)
print("PARROTLET-A LOADED")
print("=" * 70)
print("Model class:", type(model))
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)

## 3. Patched transcription function
Same logic as the model's own `transcribe()`, with one fix: casts the audio features to the encoder's dtype (bf16) before the forward pass, instead of upcasting the whole model to fp32.

In [ ]:
import torch

def transcribe_fixed(model, audio, orig_sr, max_new_tokens=256, repetition_penalty=1.2, **gen_kwargs):
    device = next(model.parameters()).device
    model_dtype = next(model.encoder.parameters()).dtype  # bf16

    prompt = model.get_prompt()
    input_ids = torch.tensor(model.tokenizer(prompt, add_special_tokens=False)['input_ids'])
    input_attention_mask = torch.ones_like(input_ids)

    audio_token = model.tokenizer.convert_tokens_to_ids(model.audio_token)
    audio_pos = input_ids.tolist().index(audio_token)

    input_ids = input_ids.unsqueeze(0).to(device)
    input_attention_mask = input_attention_mask.unsqueeze(0).to(device)

    processed_audio = model.preprocess_audio(audio, orig_sr)

    audio_features = model.processor.feature_extractor(
        [processed_audio], sampling_rate=model.sampling_rate, return_tensors="pt"
    ).input_features
    audio_features = audio_features.to(device=device, dtype=model_dtype)  # the fix

    with torch.no_grad():
        audio_embeddings = model.encoder(audio_features).last_hidden_state
        projected_audio_embeddings = model.projector(audio_embeddings)

    input_embeddings = model.decoder.get_input_embeddings()(input_ids)
    batch_size, input_seq_len, embed_dim = input_embeddings.shape
    audio_seq_len = projected_audio_embeddings.shape[1]

    max_combined_len = input_seq_len + audio_seq_len - 1
    combined_embeddings = torch.zeros(batch_size, max_combined_len, embed_dim, device=device, dtype=input_embeddings.dtype)
    combined_attention_mask = torch.zeros(batch_size, max_combined_len, device=device, dtype=input_attention_mask.dtype)

    combined_embeddings[:, :audio_pos] = input_embeddings[:, :audio_pos]
    combined_attention_mask[:, :audio_pos] = input_attention_mask[:, :audio_pos]
    combined_embeddings[:, audio_pos:audio_pos+audio_seq_len] = projected_audio_embeddings
    combined_attention_mask[:, audio_pos:audio_pos+audio_seq_len] = 1

    suffix_start = audio_pos + 1
    suffix_len = input_seq_len - suffix_start
    out_start = audio_pos + audio_seq_len
    combined_embeddings[:, out_start:out_start+suffix_len] = input_embeddings[:, suffix_start:]
    combined_attention_mask[:, out_start:out_start+suffix_len] = input_attention_mask[:, suffix_start:]

    default_gen_kwargs = {
        'max_new_tokens': max_new_tokens,
        'do_sample': False,
        'repetition_penalty': repetition_penalty,
        'pad_token_id': model.tokenizer.pad_token_id,
        'eos_token_id': model.tokenizer.eos_token_id,
    }
    default_gen_kwargs.update(gen_kwargs)

    with torch.no_grad():
        outputs = model.decoder.generate(
            inputs_embeds=combined_embeddings,
            attention_mask=combined_attention_mask,
            **default_gen_kwargs
        )

    return model.tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print("transcribe_fixed() ready.")

## 4. Upload audio and transcribe
Upload an English recording (wav/mp3/etc). It's resampled to the model's required 16kHz before transcription, sidestepping the library's internal resampling bug.

In [ ]:
import librosa
from google.colab import files

uploaded = files.upload()
file_path = list(uploaded.keys())[0]

target_sr = model.sampling_rate  # 16000
audio_array, sr = librosa.load(file_path, sr=target_sr)

print(f"Loaded '{file_path}' — {len(audio_array)/sr:.1f}s at {sr}Hz")

transcript = transcribe_fixed(model, audio_array, sr)

print("=" * 70)
print("TRANSCRIPT")
print("=" * 70)
print(transcript)

## 5. (Optional) Save transcript to a text file

In [ ]:
out_name = file_path.rsplit(".", 1)[0] + "_transcript.txt"
with open(out_name, "w", encoding="utf-8") as f:
    f.write(transcript)

print(f"Saved to {out_name}")

from google.colab import files as colab_files
colab_files.download(out_name)